# UTEC CS5363 - Planificación y Toma de Decisiones en IA (2026-2)
## Proyecto 2: Pipeline Generativo con ControlNet Canny y Servidor GPU Remoto

**Grupo 7:**
- José Guillermo Gálvez Pacori
- Matías Alonso Meneses Roncal
- Alejandro Joel Ore García

Este notebook permite ejecutar el **Nivel Intermedio (ControlNet + Stable Diffusion 1.5)** en una GPU gratuita de Google Colab (T4) o en una Máquina Virtual con CUDA, exponiendo un endpoint FastAPI para conectarse en tiempo real con la aplicación Streamlit.

### 1. Instalación de Dependencias con Soporte GPU

In [ ]:
!pip install -q diffusers transformers accelerate controlnet_aux safetensors opencv-python pillow fastapi uvicorn pyngrok

### 2. Clonar Repositorio del Proyecto

In [ ]:
!git clone https://github.com/KarTaGo124/Planning-Decision-Making-AI.git
%cd Planning-Decision-Making-AI/Project-II

### 3. Iniciar Servidor GPU con Túnel Público para Streamlit
Este comando levantará el servidor FastAPI en el puerto 8000 y creará un túnel público gratuito usando Cloudflare o ngrok para pegarlo en la barra lateral de su Streamlit local.

In [ ]:
# Descargar binario de Cloudflare Tunnel (cloudflared) para generar URL publica sin costo ni registro
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import subprocess
import time

# Iniciar el servidor FastAPI en segundo plano
server_process = subprocess.Popen(["python", "-m", "uvicorn", "src.gpu_server:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

# Iniciar el tunel Cloudflare hacia el puerto 8000
!cloudflared tunnel --url http://localhost:8000

### 4. Generación Directa de Variaciones en Notebook (Alternativa Offline)

In [ ]:
import torch
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo GPU activo: {device}")

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_canny",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
if device == "cuda":
    pipe.enable_model_cpu_offload()
print("Pipeline de ControlNet cargado con exito!")